# Problema de los pentonimios

Un pentonimio es una figura geométrica compuesta por 5 cuadrados
en este notebook lo que se hace es un algoritmo que ubique estas figuras (que están definidas al inicio de cada código) en un tablero 4x5 y 5x5, primero con backtracking y luego con un algoritmo genético

In [ ]:


import copy
from itertools import permutations

FILAS = 4
COLS = 5

PIEZAS = {
    # Roja: forma P
    #  X .
    #  X X
    #  X X
    'R': [(0,0), (1,0), (1,1), (2,0), (2,1)],

    # Azul:
    #  . . X X
    #  X X X .
    'A': [(0,2), (0,3), (1,0), (1,1), (1,2)],

    # Amarillo Oscuro:
    #  . X X X X
    #  . . X . .
    'D': [(0,1), (0,2), (0,3), (0,4), (1,2)],

    # Amarillo Claro (CORREGIDO): forma C / U lateral
    #  X X
    #  . X
    #  X X
    'C': [(0,0), (0,1), (1,1), (2,0), (2,1)],
}

for nombre, coords in PIEZAS.items():
    assert len(coords) == 5, f"¡ERROR! Pieza '{nombre}' tiene {len(coords)} celdas, debe tener 5!"
total = sum(len(c) for c in PIEZAS.values())
assert total == 20, f"¡ERROR! Total de celdas = {total}, debe ser 20 (4×5)"
print("✓ Verificación OK: 4 piezas × 5 celdas = 20 celdas (4×5)\n")

# ---------------------------------------------------------------
# Transformaciones (rotaciones y reflexiones)
# ---------------------------------------------------------------
def normalizar(pieza):
    min_f = min(c[0] for c in pieza)
    min_c = min(c[1] for c in pieza)
    return tuple(sorted([(f - min_f, c - min_c) for f, c in pieza]))

def rotar_90(pieza):
    return [(c, -f) for f, c in pieza]

def reflejar(pieza):
    return [(f, -c) for f, c in pieza]

def generar_orientaciones(pieza):
    orientaciones = set()
    actual = pieza
    for _ in range(4):
        orientaciones.add(normalizar(actual))
        orientaciones.add(normalizar(reflejar(actual)))
        actual = rotar_90(actual)
    return list(orientaciones)

# ---------------------------------------------------------------
# Posiciones válidas en el tablero
# ---------------------------------------------------------------
def generar_posiciones(pieza_coords):
    posiciones = []
    orientaciones = generar_orientaciones(pieza_coords)
    for orientacion in orientaciones:
        max_f = max(c[0] for c in orientacion)
        max_c = max(c[1] for c in orientacion)
        for df in range(FILAS - max_f):
            for dc in range(COLS - max_c):
                posicion = tuple(sorted([(f + df, c + dc) for f, c in orientacion]))
                posiciones.append(posicion)
    return list(set(posiciones))

# ---------------------------------------------------------------
# Backtracking
# ---------------------------------------------------------------
def resolver(tablero, piezas_restantes, posiciones_por_pieza, soluciones):
    if not piezas_restantes:
        soluciones.append(copy.deepcopy(tablero))
        return

    # Encontrar la primera celda vacía para podar rápidamente
    primera_vacia = None
    for f in range(FILAS):
        for c in range(COLS):
            if tablero[f][c] == '.':
                primera_vacia = (f, c)
                break
        if primera_vacia:
            break

    pieza_nombre = piezas_restantes[0]
    restantes = piezas_restantes[1:]

    for posicion in posiciones_por_pieza[pieza_nombre]:
        if primera_vacia and primera_vacia not in posicion:
            continue

        if all(tablero[f][c] == '.' for f, c in posicion):
            for f, c in posicion:
                tablero[f][c] = pieza_nombre
            resolver(tablero, restantes, posiciones_por_pieza, soluciones)
            for f, c in posicion:
                tablero[f][c] = '.'

def encontrar_todas_las_soluciones():
    posiciones_por_pieza = {}
    for nombre, coords in PIEZAS.items():
        posiciones_por_pieza[nombre] = generar_posiciones(coords)
        orientaciones = generar_orientaciones(coords)
        print(f"  Pieza '{nombre}': {len(orientaciones)} orientaciones únicas, "
              f"{len(posiciones_por_pieza[nombre])} posiciones en tablero")

    print("\n  Probando todas las combinaciones...\n")

    todas_soluciones = []
    nombres_piezas = list(PIEZAS.keys())
    soluciones_unicas = set()

    for perm in permutations(nombres_piezas):
        tablero = [['.' for _ in range(COLS)] for _ in range(FILAS)]
        soluciones = []
        resolver(tablero, list(perm), posiciones_por_pieza, soluciones)
        for sol in soluciones:
            clave = tuple(tuple(fila) for fila in sol)
            if clave not in soluciones_unicas:
                soluciones_unicas.add(clave)
                todas_soluciones.append(sol)

    return todas_soluciones

# ---------------------------------------------------------------
# Visualización
# ---------------------------------------------------------------
def imprimir_tablero(tablero, numero):
    print(f"\n  ╔═══{'═══╦═══'*(COLS-1)}═══╗")
    print(f"  ║         SOLUCIÓN #{numero:<3}              ║")
    print(f"  ╠═══{'═══╬═══'*(COLS-1)}═══╣")
    for i, fila in enumerate(tablero):
        print("  ║", end="")
        for celda in fila:
            print(f" {celda} ║", end="")
        print()
        if i < FILAS - 1:
            print(f"  ╠═══{'═══╬═══'*(COLS-1)}═══╣")
    print(f"  ╚═══{'═══╩═══'*(COLS-1)}═══╝")
    print("  R=Roja  A=Azul  D=Amar.Oscuro  C=Amar.Claro")

# ---------------------------------------------------------------
# Main
# ---------------------------------------------------------------
if __name__ == "__main__":
    print("╔══════════════════════════════════════════════╗")
    print("║    SOLVER DE PENTOMINÓS - TABLERO 4×5  v3   ║")
    print("║    4 piezas × 5 celdas = 20 (completo)      ║")
    print("╚══════════════════════════════════════════════╝\n")

    soluciones = encontrar_todas_las_soluciones()

    if soluciones:
        print(f"  ┌────────────────────────────────────┐")
        print(f"  │  ¡Se encontraron {len(soluciones):>3} solución(es)!  │")
        print(f"  └────────────────────────────────────┘")
        for i, sol in enumerate(soluciones, 1):
            imprimir_tablero(sol, i)
    else:
        print("  ⚠ No se encontraron soluciones.")
        print("  Revisa las formas de las piezas.")

    print(f"\n  ✅ Búsqueda completada.")

✓ Verificación OK: 4 piezas × 5 celdas = 20 celdas (4×5)

╔══════════════════════════════════════════════╗
║    SOLVER DE PENTOMINÓS - TABLERO 4×5  v3   ║
║    4 piezas × 5 celdas = 20 (completo)      ║
╚══════════════════════════════════════════════╝

  Pieza 'R': 8 orientaciones únicas, 68 posiciones en tablero
  Pieza 'A': 8 orientaciones únicas, 40 posiciones en tablero
  Pieza 'D': 8 orientaciones únicas, 40 posiciones en tablero
  Pieza 'C': 4 orientaciones únicas, 34 posiciones en tablero

  Probando todas las combinaciones...

  ┌────────────────────────────────────┐
  │  ¡Se encontraron   4 solución(es)!  │
  └────────────────────────────────────┘

  ╔══════╦══════╦══════╦══════╦══════╗
  ║         SOLUCIÓN #1                ║
  ╠══════╬══════╬══════╬══════╬══════╣
  ║ R ║ D ║ D ║ D ║ D ║
  ╠══════╬══════╬══════╬══════╬══════╣
  ║ R ║ R ║ D ║ C ║ C ║
  ╠══════╬══════╬══════╬══════╬══════╣
  ║ R ║ R ║ A ║ A ║ C ║
  ╠══════╬══════╬══════╬══════╬══════╣
  ║ A ║ A ║ A ║ C ║ C ║
  

In [ ]:
#Algoritmo genético

import random
import copy
import time

FILAS = 4
COLS = 5

PIEZAS = {
    'R': [(0,0), (1,0), (1,1), (2,0), (2,1)],
    'A': [(0,2), (0,3), (1,0), (1,1), (1,2)],
    'D': [(0,1), (0,2), (0,3), (0,4), (1,2)],
    'C': [(0,0), (0,1), (1,1), (2,0), (2,1)],
}

NOMBRES_PIEZAS = list(PIEZAS.keys())

NOMBRES_COMPLETOS = {
    'R': 'Roja',
    'A': 'Azul',
    'D': 'Amar.Oscuro',
    'C': 'Amar.Claro',
}

TAMANO_POBLACION   = 300
MAX_GENERACIONES   = 5000
PROB_CRUCE         = 0.85
PROB_MUTACION      = 0.40
ELITISMO           = 10       # Mejores individuos que pasan directo
TORNEO_K           = 5        # Tamaño del torneo de selección
ESTANCAMIENTO_MAX  = 150      # Generaciones sin mejora antes de reiniciar

# ---------------------------------------------------------------
# Transformaciones geométricas
# ---------------------------------------------------------------
def normalizar(pieza):
    min_f = min(c[0] for c in pieza)
    min_c = min(c[1] for c in pieza)
    return tuple(sorted([(f - min_f, c - min_c) for f, c in pieza]))

def rotar_90(pieza):
    return [(c, -f) for f, c in pieza]

def reflejar(pieza):
    return [(f, -c) for f, c in pieza]

def generar_orientaciones(pieza):
    orientaciones = set()
    actual = list(pieza)
    for _ in range(4):
        orientaciones.add(normalizar(actual))
        orientaciones.add(normalizar(reflejar(actual)))
        actual = rotar_90(actual)
    return list(orientaciones)

ORIENTACIONES = {}
for nombre, coords in PIEZAS.items():
    ORIENTACIONES[nombre] = generar_orientaciones(coords)

print("╔══════════════════════════════════════════════════╗")
print("║  SOLVER GENÉTICO DE PENTOMINÓS - TABLERO 4×5    ║")
print("╚══════════════════════════════════════════════════╝\n")
print("Orientaciones por pieza:")
for nombre in NOMBRES_PIEZAS:
    print(f"  Pieza '{nombre}' ({NOMBRES_COMPLETOS[nombre]}): "
          f"{len(ORIENTACIONES[nombre])} orientaciones únicas")
print()

# ---------------------------------------------------------------
# Cromosoma / Individuo
# ---------------------------------------------------------------
# Un individuo se representa como una lista de 4 genes (uno por pieza).
# Cada gen = (nombre_pieza, indice_orientacion, fila_offset, col_offset)
#
# El orden de la lista define el orden de colocación (prioridad).
# Las piezas colocadas primero "ganan" las celdas en caso de conflicto.

def crear_individuo():
    """Crea un individuo aleatorio."""
    genes = []
    orden = list(NOMBRES_PIEZAS)
    random.shuffle(orden)
    for nombre in orden:
        idx_orient = random.randint(0, len(ORIENTACIONES[nombre]) - 1)
        orientacion = ORIENTACIONES[nombre][idx_orient]
        max_f = max(c[0] for c in orientacion)
        max_c = max(c[1] for c in orientacion)
        fila = random.randint(0, FILAS - 1 - max_f)
        col  = random.randint(0, COLS  - 1 - max_c)
        genes.append((nombre, idx_orient, fila, col))
    return genes

def decodificar(individuo):
    """
    Coloca las piezas en el tablero en el orden del cromosoma.
    Retorna el tablero y cuántas celdas se cubrieron sin solapamiento.
    """
    tablero = [['.' for _ in range(COLS)] for _ in range(FILAS)]
    celdas_cubiertas = 0
    celdas_solapadas = 0

    for (nombre, idx_orient, fila_off, col_off) in individuo:
        orientacion = ORIENTACIONES[nombre][idx_orient]
        for (df, dc) in orientacion:
            f = fila_off + df
            c = col_off + dc
            if 0 <= f < FILAS and 0 <= c < COLS:
                if tablero[f][c] == '.':
                    tablero[f][c] = nombre
                    celdas_cubiertas += 1
                else:
                    celdas_solapadas += 1
            else:
                # Fuera del tablero (no debería pasar por construcción)
                celdas_solapadas += 1

    return tablero, celdas_cubiertas, celdas_solapadas

def fitness(individuo):
    """
    Fitness = celdas cubiertas sin solapamiento - penalización por solapamiento.
    Máximo posible = 20 (solución perfecta).
    """
    _, cubiertas, solapadas = decodificar(individuo)
    return cubiertas - (solapadas * 2)

# ---------------------------------------------------------------
# Selección por Torneo
# ---------------------------------------------------------------
def seleccion_torneo(poblacion, fitnesses):
    participantes = random.sample(range(len(poblacion)), TORNEO_K)
    mejor = max(participantes, key=lambda i: fitnesses[i])
    return copy.deepcopy(poblacion[mejor])

# ---------------------------------------------------------------
# Cruce (Crossover)
# ---------------------------------------------------------------
def cruce(padre1, padre2):
    """
    Cruce basado en orden: para cada pieza, toma el gen de un padre u otro.
    Se asegura de mantener cada pieza exactamente una vez.
    """
    if random.random() > PROB_CRUCE:
        return copy.deepcopy(padre1), copy.deepcopy(padre2)

    # Mapear nombre_pieza -> gen para cada padre
    mapa1 = {gen[0]: gen for gen in padre1}
    mapa2 = {gen[0]: gen for gen in padre2}

    # Orden de piezas de cada padre
    orden1 = [gen[0] for gen in padre1]
    orden2 = [gen[0] for gen in padre2]

    # Punto de cruce para el orden
    punto = random.randint(1, len(NOMBRES_PIEZAS) - 1)

    # Hijo 1: orden del padre1 hasta punto, luego padre2
    hijo1_orden = orden1[:punto]
    for p in orden2:
        if p not in hijo1_orden:
            hijo1_orden.append(p)

    # Hijo 2: orden del padre2 hasta punto, luego padre1
    hijo2_orden = orden2[:punto]
    for p in orden1:
        if p not in hijo2_orden:
            hijo2_orden.append(p)

    # Para los parámetros (orientación, posición), mezclar de ambos padres
    hijo1 = []
    hijo2 = []
    for nombre in hijo1_orden:
        # 50% de probabilidad de tomar parámetros de padre1 o padre2
        if random.random() < 0.5:
            hijo1.append(copy.deepcopy(mapa1[nombre]))
        else:
            hijo1.append(copy.deepcopy(mapa2[nombre]))

    for nombre in hijo2_orden:
        if random.random() < 0.5:
            hijo2.append(copy.deepcopy(mapa2[nombre]))
        else:
            hijo2.append(copy.deepcopy(mapa1[nombre]))

    return hijo1, hijo2

# ---------------------------------------------------------------
# Mutación
# ---------------------------------------------------------------
def mutar(individuo):
    """Aplica mutaciones aleatorias a un individuo."""
    individuo = copy.deepcopy(individuo)

    for i in range(len(individuo)):
        if random.random() < PROB_MUTACION:
            nombre, idx_orient, fila, col = individuo[i]
            tipo_mutacion = random.random()

            if tipo_mutacion < 0.35:
                # Cambiar orientación
                nuevo_idx = random.randint(0, len(ORIENTACIONES[nombre]) - 1)
                orientacion = ORIENTACIONES[nombre][nuevo_idx]
                max_f = max(c[0] for c in orientacion)
                max_c = max(c[1] for c in orientacion)
                # Ajustar posición si la nueva orientación no cabe
                fila = min(fila, FILAS - 1 - max_f)
                col  = min(col,  COLS  - 1 - max_c)
                individuo[i] = (nombre, nuevo_idx, fila, col)

            elif tipo_mutacion < 0.70:
                # Cambiar posición
                orientacion = ORIENTACIONES[nombre][idx_orient]
                max_f = max(c[0] for c in orientacion)
                max_c = max(c[1] for c in orientacion)
                nueva_fila = random.randint(0, FILAS - 1 - max_f)
                nueva_col  = random.randint(0, COLS  - 1 - max_c)
                individuo[i] = (nombre, idx_orient, nueva_fila, nueva_col)

            else:
                # Cambiar orientación Y posición completamente
                nuevo_idx = random.randint(0, len(ORIENTACIONES[nombre]) - 1)
                orientacion = ORIENTACIONES[nombre][nuevo_idx]
                max_f = max(c[0] for c in orientacion)
                max_c = max(c[1] for c in orientacion)
                nueva_fila = random.randint(0, FILAS - 1 - max_f)
                nueva_col  = random.randint(0, COLS  - 1 - max_c)
                individuo[i] = (nombre, nuevo_idx, nueva_fila, nueva_col)

    # Mutación de orden (swap de dos piezas)
    if random.random() < 0.30:
        i, j = random.sample(range(len(individuo)), 2)
        individuo[i], individuo[j] = individuo[j], individuo[i]

    return individuo

# ---------------------------------------------------------------
# Visualización
# ---------------------------------------------------------------
def imprimir_tablero(tablero, titulo=""):
    if titulo:
        print(f"  {titulo}")
    print("  +" + "---+"*COLS)
    for fila in tablero:
        print("  |", end="")
        for celda in fila:
            print(f" {celda} |", end="")
        print()
        print("  +" + "---+"*COLS)
    print("  R=Roja  A=Azul  D=Amar.Oscuro  C=Amar.Claro")

def imprimir_individuo(individuo, etiqueta=""):
    """Muestra la configuración de un individuo."""
    if etiqueta:
        print(f"  {etiqueta}")
    for i, (nombre, idx_orient, fila, col) in enumerate(individuo):
        print(f"    Pieza {i+1}: {NOMBRES_COMPLETOS[nombre]:12s} | "
              f"Orientación #{idx_orient:<2d} | Posición: ({fila},{col})")

# ---------------------------------------------------------------
# Algoritmo Genético Principal
# ---------------------------------------------------------------
def algoritmo_genetico():
    print(f"{'─'*55}")
    print(f"  PARÁMETROS:")
    print(f"    Población:          {TAMANO_POBLACION}")
    print(f"    Máx. generaciones:  {MAX_GENERACIONES}")
    print(f"    Prob. cruce:        {PROB_CRUCE}")
    print(f"    Prob. mutación:     {PROB_MUTACION}")
    print(f"    Elitismo:           {ELITISMO} mejores")
    print(f"    Torneo K:           {TORNEO_K}")
    print(f"    Reset estancamiento:{ESTANCAMIENTO_MAX} gen.")
    print(f"{'─'*55}\n")

    # Paso 1: Población inicial
    print("  PASO 1: Generando población inicial...\n")
    poblacion = [crear_individuo() for _ in range(TAMANO_POBLACION)]
    fitnesses = [fitness(ind) for ind in poblacion]

    mejor_global_fit = max(fitnesses)
    mejor_global_ind = copy.deepcopy(poblacion[fitnesses.index(mejor_global_fit)])

    print(f"    Población inicial creada: {TAMANO_POBLACION} individuos")
    print(f"    Mejor fitness inicial: {mejor_global_fit}/20")
    print(f"    Fitness promedio:      {sum(fitnesses)/len(fitnesses):.2f}/20\n")

    tablero_inicial, _, _ = decodificar(mejor_global_ind)
    imprimir_tablero(tablero_inicial, "Mejor individuo de la generación 0:")
    print()

    # Paso 2: Evolución
    print(f"  PASO 2: Iniciando evolución...\n")
    print(f"  {'Gen':>5s} │ {'Mejor':>5s} │ {'Promedio':>8s} │ {'Peor':>5s} │ Evento")
    print(f"  {'─'*5}─┼─{'─'*5}─┼─{'─'*8}─┼─{'─'*5}─┼─{'─'*25}")

    generaciones_sin_mejora = 0
    historial = []
    inicio = time.time()

    for gen in range(1, MAX_GENERACIONES + 1):
        # --- Selección + Cruce + Mutación ---
        nueva_poblacion = []

        # Elitismo: los mejores pasan directo
        indices_elite = sorted(range(len(fitnesses)),
                               key=lambda i: fitnesses[i], reverse=True)[:ELITISMO]
        for idx in indices_elite:
            nueva_poblacion.append(copy.deepcopy(poblacion[idx]))

        # Generar el resto de la población
        while len(nueva_poblacion) < TAMANO_POBLACION:
            padre1 = seleccion_torneo(poblacion, fitnesses)
            padre2 = seleccion_torneo(poblacion, fitnesses)
            hijo1, hijo2 = cruce(padre1, padre2)
            hijo1 = mutar(hijo1)
            hijo2 = mutar(hijo2)
            nueva_poblacion.append(hijo1)
            if len(nueva_poblacion) < TAMANO_POBLACION:
                nueva_poblacion.append(hijo2)

        poblacion = nueva_poblacion
        fitnesses = [fitness(ind) for ind in poblacion]

        mejor_fit = max(fitnesses)
        peor_fit  = min(fitnesses)
        prom_fit  = sum(fitnesses) / len(fitnesses)
        mejor_idx = fitnesses.index(mejor_fit)
        mejor_ind = poblacion[mejor_idx]

        evento = ""

        # ¿Mejora global?
        if mejor_fit > mejor_global_fit:
            mejor_global_fit = mejor_fit
            mejor_global_ind = copy.deepcopy(mejor_ind)
            generaciones_sin_mejora = 0
            evento = f"★ ¡NUEVA MEJOR! fitness={mejor_fit}"
        else:
            generaciones_sin_mejora += 1

        # Registro en historial
        historial.append({
            'gen': gen,
            'mejor': mejor_fit,
            'promedio': prom_fit,
            'peor': peor_fit,
            'evento': evento
        })

        # Imprimir progreso cada 25 generaciones o en eventos importantes
        if gen % 25 == 0 or evento or gen == 1:
            print(f"  {gen:>5d} │ {mejor_fit:>5d} │ {prom_fit:>8.2f} │ {peor_fit:>5d} │ {evento}")

        # ¿Solución encontrada? (fitness = 20)
        if mejor_global_fit == 20:
            tiempo = time.time() - inicio
            print(f"\n  {'='*55}")
            print(f"  ¡¡¡ SOLUCIÓN ENCONTRADA en generación {gen}!!!")
            print(f"  Tiempo: {tiempo:.3f} segundos")
            print(f"  {'='*55}\n")

            print("  PASO 3: Detalles de la solución\n")
            imprimir_individuo(mejor_global_ind, "Cromosoma ganador:")
            print()
            tablero_final, cubiertas, solapadas = decodificar(mejor_global_ind)
            imprimir_tablero(tablero_final, "TABLERO SOLUCIÓN:")
            print(f"\n    Celdas cubiertas: {cubiertas}/20")
            print(f"    Solapamientos:    {solapadas}")

            # Resumen evolutivo
            print(f"\n  PASO 4: Resumen del proceso evolutivo\n")
            print(f"    Generaciones totales:  {gen}")
            print(f"    Población por gen.:    {TAMANO_POBLACION}")
            print(f"    Evaluaciones totales:  {gen * TAMANO_POBLACION}")
            print(f"    Tiempo total:          {tiempo:.3f}s")

            # Mostrar hitos
            hitos = [h for h in historial if h['evento']]
            if hitos:
                print(f"\n    Hitos de mejora:")
                for h in hitos:
                    print(f"      Gen {h['gen']:>5d}: {h['evento']}")

            return mejor_global_ind, historial

        # Reset por estancamiento
        if generaciones_sin_mejora >= ESTANCAMIENTO_MAX:
            evento_reset = f"⟳ RESET (estancado {ESTANCAMIENTO_MAX} gen.)"
            print(f"  {gen:>5d} │ {mejor_fit:>5d} │ {prom_fit:>8.2f} │ "
                  f"{peor_fit:>5d} │ {evento_reset}")

            # Mantener al mejor global y regenerar el resto
            poblacion = [crear_individuo() for _ in range(TAMANO_POBLACION - 1)]
            poblacion.append(copy.deepcopy(mejor_global_ind))
            fitnesses = [fitness(ind) for ind in poblacion]
            generaciones_sin_mejora = 0

    # Si no encontró solución
    tiempo = time.time() - inicio
    print(f"\n  ⚠ No se encontró solución en {MAX_GENERACIONES} generaciones.")
    print(f"  Mejor fitness alcanzado: {mejor_global_fit}/20")
    print(f"  Tiempo: {tiempo:.3f}s\n")

    tablero_mejor, _, _ = decodificar(mejor_global_ind)
    imprimir_tablero(tablero_mejor, "Mejor tablero encontrado (incompleto):")

    return mejor_global_ind, historial

# ---------------------------------------------------------------
# Ejecución
# ---------------------------------------------------------------
if __name__ == "__main__":
    print()
    random.seed()  # Semilla aleatoria para cada ejecución
    solucion, historial = algoritmo_genetico()
    print(f"\n  ✅ Ejecución finalizada.")

╔══════════════════════════════════════════════════╗
║  SOLVER GENÉTICO DE PENTOMINÓS - TABLERO 4×5    ║
╚══════════════════════════════════════════════════╝

Orientaciones por pieza:
  Pieza 'R' (Roja): 8 orientaciones únicas
  Pieza 'A' (Azul): 8 orientaciones únicas
  Pieza 'D' (Amar.Oscuro): 8 orientaciones únicas
  Pieza 'C' (Amar.Claro): 4 orientaciones únicas


───────────────────────────────────────────────────────
  PARÁMETROS:
    Población:          300
    Máx. generaciones:  5000
    Prob. cruce:        0.85
    Prob. mutación:     0.4
    Elitismo:           10 mejores
    Torneo K:           5
    Reset estancamiento:150 gen.
───────────────────────────────────────────────────────

  PASO 1: Generando población inicial...

    Población inicial creada: 300 individuos
    Mejor fitness inicial: 14/20
    Fitness promedio:      -1.99/20

  Mejor individuo de la generación 0:
  +---+---+---+---+---+
  | A | D | D | D | D |
  +---+---+---+---+---+
  | A | . | R | R | R |
  +

In [ ]:


import random
import copy
import time

# ---------------------------------------------------------------
# Configuración del tablero y piezas
# ---------------------------------------------------------------
FILAS = 5
COLS = 5

PIEZAS = {
    'R': [(0,0), (1,0), (1,1), (2,0), (2,1)],
    'A': [(0,2), (0,3), (1,0), (1,1), (1,2)],
    'D': [(0,1), (0,2), (0,3), (0,4), (1,2)],
    'C': [(0,0), (0,1), (1,1), (2,0), (2,1)],
    'G': [(4,2), (4,3), (4,4), (3,4), (2,4)]
}

NOMBRES_PIEZAS = list(PIEZAS.keys())

NOMBRES_COMPLETOS = {
    'R': 'Roja',
    'A': 'Azul',
    'D': 'Amar.Oscuro',
    'C': 'Amar.Claro',
    'G': 'Gris',
}

TAMANO_POBLACION   = 300
MAX_GENERACIONES   = 5000
PROB_CRUCE         = 0.85
PROB_MUTACION      = 0.40
ELITISMO           = 10       # Mejores individuos que pasan directo
TORNEO_K           = 5        # Tamaño del torneo de selección
ESTANCAMIENTO_MAX  = 150      # Generaciones sin mejora antes de reiniciar

# ---------------------------------------------------------------
# Transformaciones geométricas
# ---------------------------------------------------------------
def normalizar(pieza):
    min_f = min(c[0] for c in pieza)
    min_c = min(c[1] for c in pieza)
    return tuple(sorted([(f - min_f, c - min_c) for f, c in pieza]))

def rotar_90(pieza):
    return [(c, -f) for f, c in pieza]

def reflejar(pieza):
    return [(f, -c) for f, c in pieza]

def generar_orientaciones(pieza):
    orientaciones = set()
    actual = list(pieza)
    for _ in range(4):
        orientaciones.add(normalizar(actual))
        orientaciones.add(normalizar(reflejar(actual)))
        actual = rotar_90(actual)
    return list(orientaciones)

ORIENTACIONES = {}
for nombre, coords in PIEZAS.items():
    ORIENTACIONES[nombre] = generar_orientaciones(coords)

print("╔══════════════════════════════════════════════════╗")
print("║  SOLVER GENÉTICO DE PENTOMINÓS - TABLERO 5×5    ║")
print("╚══════════════════════════════════════════════════╝\n")
print("Orientaciones por pieza:")
for nombre in NOMBRES_PIEZAS:
    print(f"  Pieza '{nombre}' ({NOMBRES_COMPLETOS[nombre]}): "
          f"{len(ORIENTACIONES[nombre])} orientaciones únicas")
print()

# ---------------------------------------------------------------
# Cromosoma / Individuo
# ---------------------------------------------------------------
def crear_individuo():
    """Crea un individuo aleatorio."""
    genes = []
    orden = list(NOMBRES_PIEZAS)
    random.shuffle(orden)
    for nombre in orden:
        idx_orient = random.randint(0, len(ORIENTACIONES[nombre]) - 1)
        orientacion = ORIENTACIONES[nombre][idx_orient]
        max_f = max(c[0] for c in orientacion)
        max_c = max(c[1] for c in orientacion)
        fila = random.randint(0, FILAS - 1 - max_f)
        col  = random.randint(0, COLS  - 1 - max_c)
        genes.append((nombre, idx_orient, fila, col))
    return genes

# ---------------------------------------------------------------
# Decodificación y Fitness
# ---------------------------------------------------------------
def decodificar(individuo):
    """
    Coloca las piezas en el tablero en el orden del cromosoma.
    Retorna el tablero y cuántas celdas se cubrieron sin solapamiento.
    """
    tablero = [['.' for _ in range(COLS)] for _ in range(FILAS)]
    celdas_cubiertas = 0
    celdas_solapadas = 0

    for (nombre, idx_orient, fila_off, col_off) in individuo:
        orientacion = ORIENTACIONES[nombre][idx_orient]
        for (df, dc) in orientacion:
            f = fila_off + df
            c = col_off + dc
            if 0 <= f < FILAS and 0 <= c < COLS:
                if tablero[f][c] == '.':
                    tablero[f][c] = nombre
                    celdas_cubiertas += 1
                else:
                    celdas_solapadas += 1
            else:
                celdas_solapadas += 1

    return tablero, celdas_cubiertas, celdas_solapadas

def fitness(individuo):
    """
    Fitness = celdas cubiertas sin solapamiento - penalización por solapamiento.
    Máximo posible = 25 (solución perfecta para 5x5).
    """
    _, cubiertas, solapadas = decodificar(individuo)
    return cubiertas - (solapadas * 2)

# ---------------------------------------------------------------
# Selección por Torneo
# ---------------------------------------------------------------
def seleccion_torneo(poblacion, fitnesses):
    participantes = random.sample(range(len(poblacion)), TORNEO_K)
    mejor = max(participantes, key=lambda i: fitnesses[i])
    return copy.deepcopy(poblacion[mejor])

# ---------------------------------------------------------------
# Cruce (Crossover)
# ---------------------------------------------------------------
def cruce(padre1, padre2):
    if random.random() > PROB_CRUCE:
        return copy.deepcopy(padre1), copy.deepcopy(padre2)

    mapa1 = {gen[0]: gen for gen in padre1}
    mapa2 = {gen[0]: gen for gen in padre2}

    orden1 = [gen[0] for gen in padre1]
    orden2 = [gen[0] for gen in padre2]

    punto = random.randint(1, len(NOMBRES_PIEZAS) - 1)

    hijo1_orden = orden1[:punto]
    for p in orden2:
        if p not in hijo1_orden:
            hijo1_orden.append(p)

    hijo2_orden = orden2[:punto]
    for p in orden1:
        if p not in hijo2_orden:
            hijo2_orden.append(p)

    hijo1 = []
    hijo2 = []
    for nombre in hijo1_orden:
        if random.random() < 0.5:
            hijo1.append(copy.deepcopy(mapa1[nombre]))
        else:
            hijo1.append(copy.deepcopy(mapa2[nombre]))

    for nombre in hijo2_orden:
        if random.random() < 0.5:
            hijo2.append(copy.deepcopy(mapa2[nombre]))
        else:
            hijo2.append(copy.deepcopy(mapa1[nombre]))

    return hijo1, hijo2

# ---------------------------------------------------------------
# Mutación
# ---------------------------------------------------------------
def mutar(individuo):
    individuo = copy.deepcopy(individuo)

    for i in range(len(individuo)):
        if random.random() < PROB_MUTACION:
            nombre, idx_orient, fila, col = individuo[i]
            tipo_mutacion = random.random()

            if tipo_mutacion < 0.35:
                nuevo_idx = random.randint(0, len(ORIENTACIONES[nombre]) - 1)
                orientacion = ORIENTACIONES[nombre][nuevo_idx]
                max_f = max(c[0] for c in orientacion)
                max_c = max(c[1] for c in orientacion)
                fila = min(fila, FILAS - 1 - max_f)
                col  = min(col,  COLS  - 1 - max_c)
                individuo[i] = (nombre, nuevo_idx, fila, col)

            elif tipo_mutacion < 0.70:
                orientacion = ORIENTACIONES[nombre][idx_orient]
                max_f = max(c[0] for c in orientacion)
                max_c = max(c[1] for c in orientacion)
                nueva_fila = random.randint(0, FILAS - 1 - max_f)
                nueva_col  = random.randint(0, COLS  - 1 - max_c)
                individuo[i] = (nombre, idx_orient, nueva_fila, nueva_col)

            else:
                nuevo_idx = random.randint(0, len(ORIENTACIONES[nombre]) - 1)
                orientacion = ORIENTACIONES[nombre][nuevo_idx]
                max_f = max(c[0] for c in orientacion)
                max_c = max(c[1] for c in orientacion)
                nueva_fila = random.randint(0, FILAS - 1 - max_f)
                nueva_col  = random.randint(0, COLS  - 1 - max_c)
                individuo[i] = (nombre, nuevo_idx, nueva_fila, nueva_col)

    if random.random() < 0.30:
        i, j = random.sample(range(len(individuo)), 2)
        individuo[i], individuo[j] = individuo[j], individuo[i]

    return individuo

# ---------------------------------------------------------------
# Visualización
# ---------------------------------------------------------------
def imprimir_tablero(tablero, titulo=""):
    if titulo:
        print(f"  {titulo}")
    print("  +" + "---+"*COLS)
    for fila in tablero:
        print("  |", end="")
        for celda in fila:
            print(f" {celda} |", end="")
        print()
        print("  +" + "---+"*COLS)
    print("  R=Roja  A=Azul  D=Amar.Oscuro  C=Amar.Claro  G=Gris")

def imprimir_individuo(individuo, etiqueta=""):
    if etiqueta:
        print(f"  {etiqueta}")
    for i, (nombre, idx_orient, fila, col) in enumerate(individuo):
        print(f"    Pieza {i+1}: {NOMBRES_COMPLETOS[nombre]:12s} | "
              f"Orientación #{idx_orient:<2d} | Posición: ({fila},{col})")

# ---------------------------------------------------------------
# Algoritmo Genético Principal
# ---------------------------------------------------------------
def algoritmo_genetico():
    print(f"{'─'*55}")
    print(f"  PARÁMETROS:")
    print(f"    Población:          {TAMANO_POBLACION}")
    print(f"    Máx. generaciones:  {MAX_GENERACIONES}")
    print(f"    Prob. cruce:        {PROB_CRUCE}")
    print(f"    Prob. mutación:     {PROB_MUTACION}")
    print(f"    Elitismo:           {ELITISMO} mejores")
    print(f"    Torneo K:           {TORNEO_K}")
    print(f"    Reset estancamiento:{ESTANCAMIENTO_MAX} gen.")
    print(f"{'─'*55}\n")

    print("  PASO 1: Generando población inicial...\n")
    poblacion = [crear_individuo() for _ in range(TAMANO_POBLACION)]
    fitnesses = [fitness(ind) for ind in poblacion]

    mejor_global_fit = max(fitnesses)
    mejor_global_ind = copy.deepcopy(poblacion[fitnesses.index(mejor_global_fit)])

    print(f"    Población inicial creada: {TAMANO_POBLACION} individuos")
    print(f"    Mejor fitness inicial: {mejor_global_fit}/25")
    print(f"    Fitness promedio:      {sum(fitnesses)/len(fitnesses):.2f}/25\n")

    tablero_inicial, _, _ = decodificar(mejor_global_ind)
    imprimir_tablero(tablero_inicial, "Mejor individuo de la generación 0:")
    print()

    print(f"  PASO 2: Iniciando evolución...\n")
    print(f"  {'Gen':>5s} │ {'Mejor':>5s} │ {'Promedio':>8s} │ {'Peor':>5s} │ Evento")
    print(f"  {'─'*5}─┼─{'─'*5}─┼─{'─'*8}─┼─{'─'*5}─┼─{'─'*25}")

    generaciones_sin_mejora = 0
    historial = []
    inicio = time.time()

    for gen in range(1, MAX_GENERACIONES + 1):
        nueva_poblacion = []

        indices_elite = sorted(range(len(fitnesses)),
                               key=lambda i: fitnesses[i], reverse=True)[:ELITISMO]
        for idx in indices_elite:
            nueva_poblacion.append(copy.deepcopy(poblacion[idx]))

        while len(nueva_poblacion) < TAMANO_POBLACION:
            padre1 = seleccion_torneo(poblacion, fitnesses)
            padre2 = seleccion_torneo(poblacion, fitnesses)
            hijo1, hijo2 = cruce(padre1, padre2)
            hijo1 = mutar(hijo1)
            hijo2 = mutar(hijo2)
            nueva_poblacion.append(hijo1)
            if len(nueva_poblacion) < TAMANO_POBLACION:
                nueva_poblacion.append(hijo2)

        poblacion = nueva_poblacion
        fitnesses = [fitness(ind) for ind in poblacion]

        mejor_fit = max(fitnesses)
        peor_fit  = min(fitnesses)
        prom_fit  = sum(fitnesses) / len(fitnesses)
        mejor_idx = fitnesses.index(mejor_fit)
        mejor_ind = poblacion[mejor_idx]

        evento = ""

        if mejor_fit > mejor_global_fit:
            mejor_global_fit = mejor_fit
            mejor_global_ind = copy.deepcopy(mejor_ind)
            generaciones_sin_mejora = 0
            evento = f"★ ¡NUEVA MEJOR! fitness={mejor_fit}"
        else:
            generaciones_sin_mejora += 1

        historial.append({
            'gen': gen,
            'mejor': mejor_fit,
            'promedio': prom_fit,
            'peor': peor_fit,
            'evento': evento
        })

        if gen % 25 == 0 or evento or gen == 1:
            print(f"  {gen:>5d} │ {mejor_fit:>5d} │ {prom_fit:>8.2f} │ {peor_fit:>5d} │ {evento}")

        # Solución perfecta es 25 (5 piezas x 5 celdas)
        if mejor_global_fit == 25:
            tiempo = time.time() - inicio
            print(f"\n  {'='*55}")
            print(f"  ¡¡¡ SOLUCIÓN ENCONTRADA en generación {gen}!!!")
            print(f"  Tiempo: {tiempo:.3f} segundos")
            print(f"  {'='*55}\n")

            print("  PASO 3: Detalles de la solución\n")
            imprimir_individuo(mejor_global_ind, "Cromosoma ganador:")
            print()
            tablero_final, cubiertas, solapadas = decodificar(mejor_global_ind)
            imprimir_tablero(tablero_final, "TABLERO SOLUCIÓN:")
            print(f"\n    Celdas cubiertas: {cubiertas}/25")
            print(f"    Solapamientos:    {solapadas}")

            print(f"\n  PASO 4: Resumen del proceso evolutivo\n")
            print(f"    Generaciones totales:  {gen}")
            print(f"    Población por gen.:    {TAMANO_POBLACION}")
            print(f"    Evaluaciones totales:  {gen * TAMANO_POBLACION}")
            print(f"    Tiempo total:          {tiempo:.3f}s")

            hitos = [h for h in historial if h['evento']]
            if hitos:
                print(f"\n    Hitos de mejora:")
                for h in hitos:
                    print(f"      Gen {h['gen']:>5d}: {h['evento']}")

            return mejor_global_ind, historial

        if generaciones_sin_mejora >= ESTANCAMIENTO_MAX:
            evento_reset = f"⟳ RESET (estancado {ESTANCAMIENTO_MAX} gen.)"
            print(f"  {gen:>5d} │ {mejor_fit:>5d} │ {prom_fit:>8.2f} │ "
                  f"{peor_fit:>5d} │ {evento_reset}")

            poblacion = [crear_individuo() for _ in range(TAMANO_POBLACION - 1)]
            poblacion.append(copy.deepcopy(mejor_global_ind))
            fitnesses = [fitness(ind) for ind in poblacion]
            generaciones_sin_mejora = 0

    tiempo = time.time() - inicio
    print(f"\n  ⚠ No se encontró solución en {MAX_GENERACIONES} generaciones.")
    print(f"  Mejor fitness alcanzado: {mejor_global_fit}/25")
    print(f"  Tiempo: {tiempo:.3f}s\n")

    tablero_mejor, _, _ = decodificar(mejor_global_ind)
    imprimir_tablero(tablero_mejor, "Mejor tablero encontrado (incompleto):")

    return mejor_global_ind, historial

# ---------------------------------------------------------------
# Ejecución
# ---------------------------------------------------------------
if __name__ == "__main__":
    print()
    random.seed()
    solucion, historial = algoritmo_genetico()
    print(f"\n  ✅ Ejecución finalizada.")

╔══════════════════════════════════════════════════╗
║  SOLVER GENÉTICO DE PENTOMINÓS - TABLERO 5×5    ║
╚══════════════════════════════════════════════════╝

Orientaciones por pieza:
  Pieza 'R' (Roja): 8 orientaciones únicas
  Pieza 'A' (Azul): 8 orientaciones únicas
  Pieza 'D' (Amar.Oscuro): 8 orientaciones únicas
  Pieza 'C' (Amar.Claro): 4 orientaciones únicas
  Pieza 'G' (Gris): 4 orientaciones únicas


───────────────────────────────────────────────────────
  PARÁMETROS:
    Población:          300
    Máx. generaciones:  5000
    Prob. cruce:        0.85
    Prob. mutación:     0.4
    Elitismo:           10 mejores
    Torneo K:           5
    Reset estancamiento:150 gen.
───────────────────────────────────────────────────────

  PASO 1: Generando población inicial...

    Población inicial creada: 300 individuos
    Mejor fitness inicial: 10/25
    Fitness promedio:      -2.59/25

  Mejor individuo de la generación 0:
  +---+---+---+---+---+
  | . | . | R | R | . |
  +---+-